# Module 03 - Cleaning research data

Goal: turn a messy synthetic learner survey into a clean analysis table, a cleaning log, and a before/after quality figure. The notebook is written for first-time Python learners: inspect first, decide second, clean third, write last.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

BASE = Path.cwd()
RAW = BASE / "data" / "raw"
OUT_TABLES = BASE / "outputs" / "tables"
OUT_FIGURES = BASE / "outputs" / "figures"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

## 1. Read the messy learner survey

Start with the raw table. Do not clean immediately; first ask what can go wrong if this table is plotted as-is.

In [ ]:
raw = pd.read_csv(RAW / "module03_messy_learner_survey.csv", keep_default_na=False)
print(raw.shape)
raw.head()

## 2. Inspect common data-quality problems

For research figures, the most dangerous problems are often quiet: duplicates, missing cells, label variants, and values stored with the wrong type.

In [ ]:
print("Duplicate learner-date keys:", raw.duplicated(subset=["learner_id", "survey_date"], keep="first").sum())
print("Activity labels:")
print(raw["activity_group"].value_counts())
print("Missing pre/post cells:")
print((raw[["pre_vocab_score", "post_vocab_score"]] == "").sum())

## 3. Define small cleaning helpers

A helper function makes the label-cleaning rule visible and reusable.

In [ ]:
def normalize_text(series):
    return (series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[_-]+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True))

activity_map = {
    "task based": "task-based",
    "form focused": "form-focused",
    "mtpe practice": "mtpe-practice",
    "contrastive mini lesson": "contrastive-mini-lesson",
}

## 4. Clean one rule at a time

Keep raw audit columns for values that may need to be explained later.

In [ ]:
exact_dup_mask = raw.duplicated(subset=["learner_id", "survey_date"], keep="first")
clean = raw.loc[~exact_dup_mask].copy()
clean["activity_group_raw"] = clean["activity_group"]
clean["activity_group"] = normalize_text(clean["activity_group"]).map(activity_map)

for col in ["pre_vocab_score", "post_vocab_score", "minutes_on_task", "satisfaction_1_5"]:
    clean[f"{col}_raw"] = clean[col]
    clean[col] = pd.to_numeric(clean[col].replace("", pd.NA), errors="coerce")

clean["survey_date_raw"] = clean["survey_date"]
clean["survey_date"] = pd.to_datetime(clean["survey_date"].replace("", pd.NA), errors="coerce", format="mixed")

invalid_post = clean["post_vocab_score"].notna() & ~clean["post_vocab_score"].between(0, 100)
invalid_minutes = clean["minutes_on_task"].notna() & (clean["minutes_on_task"] < 0)
invalid_sat = clean["satisfaction_1_5"].notna() & ~clean["satisfaction_1_5"].between(1, 5)

clean.loc[invalid_post, "post_vocab_score"] = pd.NA
clean.loc[invalid_minutes, "minutes_on_task"] = pd.NA
clean.loc[invalid_sat, "satisfaction_1_5"] = pd.NA
clean["complete_vocab_pair"] = clean[["pre_vocab_score", "post_vocab_score"]].notna().all(axis=1)
clean["gain_vocab_score"] = clean["post_vocab_score"] - clean["pre_vocab_score"]
clean.head()

## 5. Write a cleaning log

The log is part of the paper package. It records what was detected, what you decided, and why it matters for interpretation.

In [ ]:
cleaning_log = pd.DataFrame([
    ["duplicate", "duplicate learner-date key", int(exact_dup_mask.sum()), "Remove later row after checking learner_id and survey_date repeat the same observation unit."],
    ["label", "activity_group variants", raw["activity_group"].nunique(), "Normalize case, spaces, hyphens, and underscores."],
    ["missing", "pre/post vocabulary score", int((raw[["pre_vocab_score", "post_vocab_score"]] == "").sum().sum()), "Keep as NA; use complete pairs for gain."],
    ["range", "post score outside 0-100", int(invalid_post.sum()), "Set impossible score to NA and preserve raw value."],
    ["range", "minutes_on_task below zero", int(invalid_minutes.sum()), "Set impossible minutes to NA."],
    ["type", "survey_date parse failure", int(clean["survey_date"].isna().sum()), "Invalid date becomes missing date."],
], columns=["issue_type", "variable_or_rule", "detected_before", "cleaning_decision"])
cleaning_log

## 6. Export clean outputs

A reproducible workflow should regenerate the same clean dataset and log from the raw file.

In [ ]:
clean.to_csv(OUT_TABLES / "module03_cleaned_learner_survey.csv", index=False)
cleaning_log.to_csv(OUT_TABLES / "module03_student_cleaning_log.csv", index=False)
print("complete pre/post pairs:", clean["complete_vocab_pair"].sum(), "of", len(clean))

## 7. Plot before/after quality checks

This figure is not an analysis result. It is a transparency figure: it shows what the cleaning workflow fixed or documented.

In [ ]:
quality = pd.read_csv(OUT_TABLES / "module03_quality_summary.csv")
fig, ax = plt.subplots(figsize=(10, 6))
y = range(len(quality))
ax.barh(y, quality["issue_count_before"], label="Before cleaning")
ax.barh(y, quality["unresolved_after_cleaning"], label="Unresolved after cleaning")
ax.set_yticks(y, quality["quality_check"])
ax.set_xlabel("Number of unresolved problems")
ax.set_title("Before/after data-quality checks")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT_FIGURES / "module03_student_quality_check.png", dpi=200)

## 8. Draft the Methods note

Use the cleaning log to write a short, accountable Methods sentence.

In [ ]:
methods_note = (
    "The learner-survey data were screened for duplicate learner-date records, missing outcome cells, "
    "non-canonical activity labels, invalid numeric ranges, and date parsing failures. "
    "One duplicate learner-date record was removed. Activity labels were normalized with a documented map. "
    "Invalid numeric values were set to missing and raw audit columns were preserved; gain scores "
    "were calculated only for learners with complete pre/post vocabulary scores."
)
print(methods_note)